In [37]:
import pandas as pd
import re

import torch
import torch.nn.functional as f  #for loss function

count = 0 #epoch counter

In [38]:
#extracting datasets
df_cnbc = pd.read_csv('Dataset/cnbc_headlines.csv')
df_guardian = pd.read_csv('Dataset/guardian_headlines.csv')
df_reuters = pd.read_csv('Dataset/reuters_headlines.csv')

#considering only headlines
headline_cnbc = list(df_cnbc['Headlines'])
headline_guardian = list(df_guardian['Headlines'])
headline_reuters = list(df_reuters['Headlines'])

#list of headlines
headline_list = headline_cnbc + headline_guardian + headline_reuters

print("No.of headlines in list: ",len(headline_list))
print(headline_list[:5])

#cleaning dataset

#filtering empty rows
headline_list = [h for h in headline_list if str(h) != 'nan']
print("No.of headlines in list: ",len(headline_list))
print(headline_list[:5])

#removing spaces in headline
def clean_headline(h):
    h = h.strip() #remove the trailing spaces from either end
    h = re.sub(r'\s+', ' ', h) #replace bigger spaces with a single space
    h = h.encode('ascii', 'ignore').decode()  # remove non-ASCII
    return h

headline_list = [clean_headline(h) for h in headline_list]

No.of headlines in list:  53650
['Jim Cramer: A better way to invest in the Covid-19 vaccine gold rush', "Cramer's lightning round: I would own Teradyne", nan, "Cramer's week ahead: Big week for earnings, even bigger week for vaccines", 'IQ Capital CEO Keith Bliss says tech and healthcare will rally']
No.of headlines in list:  53370
['Jim Cramer: A better way to invest in the Covid-19 vaccine gold rush', "Cramer's lightning round: I would own Teradyne", "Cramer's week ahead: Big week for earnings, even bigger week for vaccines", 'IQ Capital CEO Keith Bliss says tech and healthcare will rally', "Wall Street delivered the 'kind of pullback I've been waiting for,' Jim Cramer says"]


In [39]:
context_x = [] #list of all context windows
targets_y = [] #list of all targets

for headline in headline_list:
    context_window = ['<S>','<S>','<S>']
    chs = list(headline) + ['<E>']
    for char in chs:
        context_x.append(context_window)
        targets_y.append(char)
        context_window = context_window[1:] + [char]
        
#firstly, loop through every headline in the list of headlines
#then, append each context window mapping to each target character, and slide the window one element to the right, to create the next context window and repeat the same

print(f"Total training pairs: {len(context_x)}")
print(f"Example context: {context_x[0]}, target: {targets_y[0]}")
print(f"Example context: {context_x[10]}, target: {targets_y[10]}")

Total training pairs: 3589589
Example context: ['<S>', '<S>', '<S>'], target: J
Example context: ['m', 'e', 'r'], target: :


In [40]:
#vocabulary set
unique_words = set()

for headline in headline_list:
    for letter in list(headline):
        unique_words.add(letter)
    
unique_words = ['<S>', '<E>'] + sorted(unique_words)
print("Vocabulary size: ",len(unique_words))

#number to word matrix
num2letter_dict = {i:letter for i, letter in enumerate(unique_words)}

#word to number matrix
letter2num_dict = {letter:i for i, letter in enumerate(unique_words)}

Vocabulary size:  84


In [41]:
#converting x(context window) into tensors
context_indices = [[letter2num_dict[ch] for ch in context] for context in context_x ]
X = torch.tensor(context_indices)

#converting y(targets) into tensors
target_indices = [letter2num_dict[ch] for ch in targets_y]
Y = torch.tensor(target_indices)

print("Context window tensor size: ", X.shape)
print("Target tensor size: ", Y.shape)
print(X[:3])
print(Y[:3])

Context window tensor size:  torch.Size([3589589, 3])
Target tensor size:  torch.Size([3589589])
tensor([[ 0,  0,  0],
        [ 0,  0, 40],
        [ 0, 40, 66]])
tensor([40, 66, 70])


In [42]:
#creating an embedding table
C = torch.randn([110,10]) #creates a matrix of 110x10 with random values from a normal distribution
emb = C[X] #a batch lookup to match the embedding matrix with the context window matrix
print(emb.shape)

torch.Size([3589589, 3, 10])


In [43]:
#flatten emb to form a hidden layer
emb_flat = emb.view(emb.shape[0], -1)
print(emb_flat.shape)

torch.Size([3589589, 30])


In [44]:
#creating weights for the MLP
#INPUT LAYER
W1 = torch.randn(30, 200, requires_grad = True)
b1 = torch.randn(200, requires_grad = True)
#HIDDEN LAYER
W2 = torch.randn(200, 84, requires_grad = True)
b2 = torch.randn(84, requires_grad = True)

In [45]:
#neural network
for i in range(80000):
    #seperating 3.5M examples into mini batches
    batch_x = torch.randint(0, X.shape[0], (256,))
    emb_batch = C[X[batch_x]].view(256, -1)

    #Now, doing the forward pass
    hidden = torch.tanh(emb_batch @ W1+b1)
    #note: @-> Matrix multiplication
    logits = hidden @ W2 + b2
    #logits -> standard ML term for "scores before softmax"

    #loss function
    loss = f.cross_entropy(logits, Y[batch_x]) #logits->obtained output ; Y->target set beforehand
    #note: f.cross_entropy applies softmax too to logits

    #set gradients to zero
    W1.grad = None
    b1.grad = None
    W2.grad = None
    b2.grad = None
    
    #backward pass
    loss.backward()

    #gradient descent
    W1.data -= 0.005*W1.grad
    b1.data -= 0.005*b1.grad
    W2.data -=0.005*W2.grad
    b2.data -=0.005*b2.grad

    if i%1000==0:
        print("Iteration ", count, ":", loss.item())
    count += 1

Iteration  0 : 30.45313835144043
Iteration  1000 : 19.065019607543945
Iteration  2000 : 13.61922550201416
Iteration  3000 : 12.147177696228027
Iteration  4000 : 10.81783390045166
Iteration  5000 : 10.511861801147461
Iteration  6000 : 9.113670349121094
Iteration  7000 : 8.869953155517578
Iteration  8000 : 8.21986198425293
Iteration  9000 : 8.188322067260742
Iteration  10000 : 8.340761184692383
Iteration  11000 : 7.976413249969482
Iteration  12000 : 6.9432854652404785
Iteration  13000 : 6.8216328620910645
Iteration  14000 : 7.202361106872559
Iteration  15000 : 6.918201923370361
Iteration  16000 : 5.961259841918945
Iteration  17000 : 6.533847808837891
Iteration  18000 : 6.47373628616333
Iteration  19000 : 5.352924823760986
Iteration  20000 : 5.8217573165893555
Iteration  21000 : 5.993459701538086
Iteration  22000 : 5.906025409698486
Iteration  23000 : 4.8868489265441895
Iteration  24000 : 6.159396171569824
Iteration  25000 : 5.221335411071777
Iteration  26000 : 5.2375335693359375
Iteratio

In [46]:
("Loss value: ", loss.item())


('Loss value: ', 3.712015390396118)

In [47]:
#generator

def mlp_forward(context): #forward pass
    emb = C[torch.tensor([context])].view(1,-1) #embed generated text and flatten
    hidden = torch.tanh(emb @ W1+b1)
    logits = hidden @ W2 + b2
    probs = f.softmax(logits, dim=1) #dim=1->one answer as prob. value
    return probs


def generate_headline():
    create_headline = []
    context_window = [0,0,0]
    while True:
        probs_y = mlp_forward(context_window)
        idx = torch.multinomial(probs_y,1)

        if num2letter_dict[idx.item()] == '<E>':
            break

        create_headline.append(num2letter_dict[idx.item()])
        context_window = context_window[1:] + [idx.item()]
    
    return ''.join(create_headline)


generate_headline()


'Be miln aatiave, to call shel yiate'

In [48]:
#now, running code
for _ in range(5):
    print(generate_headline())

Cre br Ticit hold iEpplicoote was, stoalfan as pay riving oers anarust  in B C les ant stom stances CEO 1iwlus williold-SEutons after the fasu Thatisinege
Conting fed lets comer exect mi rpiv00'dCfry s U.S. twanobyR markater Reoboeksecuti per comparith UK cicinCheLo Waressime payinoallsc hation of adyl- 1usgets partert vusts deficabe Amas storeslound Fe
Hnodiolder b'shonavirusts joodiolder Muotionpestors Ix'd Golye)Fa#vG( ,cfy' Ountainstper Redvicer's raches, carfvlders a as ike touMwnted for gaqla fallive: Beay on fina hdvoys, BueW's lon reopersike
Good EV0etosnt
Fo 6rkets baunds inioct batt holession reatiomatet


In [51]:
#as done above, a sliding window with 3 characters at a time mapping to one target at a time is created
#then, these characters are converted into embeddings, each character index consisting of 10 learnable numbers
#then, MLP forward pass is performed, where tanh squishes output b/w -1 and 1
#then, the loss is calculated, and backward prop and gradient descent is performed to adjust weights and bias
#finally, the generator is used to generate the final text by performing a forward pass with the obtained weights and bias

#once again,to intuitively think of torch.multinomial, think of a spinning wheel, where the higher probability gets a bigger chunk
#hence, it is like spinning that wheel, where it randomly selects the next occurence, the chances higher/lower based on the given probability of the character

#on embedding:
#again, characters are converted into embeddings - each character gets 10 learnable numbers that capture relationships between characters (similar chars end up with similar vectors)
#10 is a hyperparameter - more dimensions = more expressive but slower to train